## Notebook for selecting Few-shot examples based on quality. Extracts top and bottom LENS and Compression Ratio

In [2]:
import subprocess
import pandas as pd

N = 3 #top N examples

csv_file = "csv/M-7.csv"

df = pd.read_csv(csv_file)

In [3]:
lens_venv = "/home/c23068554/miniconda3/envs/lens_eval/bin/python"
def get_lens_scores(csv_path):
    result = subprocess.run([lens_venv, "lens_score_API.py", csv_path, "--per-row"], capture_output=True, text=True)
    lines = [l for l in result.stdout.strip().split("\n") if l.replace(".", "").replace("-", "").isdigit()]
    return list(map(float, lines))

In [4]:
df["lens"] = get_lens_scores(csv_file)
df["compression"] = df["Reference"].str.len() / df["Original"].str.len()

In [5]:
# LENS, columns renamed to be csv_read back with the same headers as SimPA
df.nlargest(N, "lens")[["Original", "Reference"]].rename(columns={"Original": "original", "Reference": "simplified"}).to_csv("fewshot_top_lens.csv", index=False)
df.nsmallest(N, "lens")[["Original", "Reference"]].rename(columns={"Original": "original", "Reference": "simplified"}).to_csv("fewshot_bottom_lens.csv", index=False)

# Compression, columns renamed to be csv_read back with the same headers as SimPA
df.nsmallest(N, "compression")[["Original", "Reference"]].rename(columns={"Original": "original", "Reference": "simplified"}).to_csv("fewshot_top_compression.csv", index=False)
df.nlargest(N, "compression")[["Original", "Reference"]].rename(columns={"Original": "original", "Reference": "simplified"}).to_csv("fewshot_bottom_compression.csv", index=False)